# Computing time-overlaps in Slater Determinant (SD), Configuration State Function (CSF), and Configuration Interaction (CI ) bases

## Table of contents <a name="toc"></a>

1. [Extracting TD-DFT data from CP2K log files](#1)

2. [Constructing the mapped SD basis and corresponding CSFs](#2)

3. [Computing SD and CSF time-overlaps](#3)

4. [Computing CI time-overlaps](#4)

   4.1. [Organizing basis and amplitudes](#4.1)
  
   4.2. [Transformation to the CI basis](#4.2)
     

## A. Learning objectives

- To read the CI information from the CP2K output of TD-DFT calculations
- To constructe the basis of Slater determinants corresponding to excitations present in the CP2K TD-DFT output
- To construct the mapped SD basis of unique determinants
- To constructe the basis of CSF states and the corresponding SD-to-CSF transformation matrices
- To compute the time-overlaps in the SD, CSF, and CI bases


## B. Use cases

- Manually construct a Slater Determinant basis
- Constructing configuration spin functions
- Spin-adaptation of Slater determinants
- Computing many-body (TD-DFT, TD-DFTB, CI) NACs
- Computing NACs using CP2K/Libra
- Working with sparse NumPy matrices

## C. Functions

- `libra_py`
  - `citools`
    - `interfaces`
      - [`configs_and_T_matrix_singlet`](#configs_and_T_matrix_singlet-1)
    - `slatdet`
      - [`excitation_phase_from_mapping`](#excitation_phase_from_mapping-1)
      - [`slater_overlap_matrix`](#slater_overlap_matrix-1)
  - `packages`
    - `cp2k`
      - `methods`
        - [`read_cp2k_tddfpt_log_file`](#read_cp2k_tddfpt_log_file-1)
        - [`read_homo_index`](#read_homo_index-1)        

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import time

from liblibra_core import *
from libra_py import units
from libra_py import data_conv
from libra_py import molden_methods
import libra_py.packages.cp2k.methods as CP2K_methods
import libra_py
import libra_py.citools.slatdet as sd
import libra_py.citools.interfaces as interfaces

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Extracting TD-DFT data from CP2K log files
<a name="1"></a>
[Back to TOC](#toc)

First, let's read the KS HOMO index from the CP2K output using `CP2K_methods.read_homo_index` function. It will retrieve two numbers for alpha and beta HOMO index if there are two different HOMO index in the CP2K logfiles.
<a name="read_homo_index-1"></a>

In [2]:
filename = "step_1200.log"
ks_homo_index = CP2K_methods.read_homo_index(filename)
print('The HOMO index is:',ks_homo_index)

The HOMO index is: 40


Now, we can read the orbital information for a selected set of orbitals using the `CP2K_methods.read_cp2k_tddfpt_log_file` function. 

It is convenient to define the set of orbitals to read in their relation to the HOMO index (which we extracted just above):
<a name="read_cp2k_tddfpt_log_file-1"></a>

In [3]:
lowest_orbital = ks_homo_index - 19
highest_orbital = ks_homo_index + 20
params_tddft = {'number_of_states': 10, 'ci_threshold': 0.05, 
                'logfile_name': filename, 'isUKS': False, 
                }
info, data = CP2K_methods.read_cp2k_tddfpt_log_file(params_tddft)

The info looks like:

In [4]:
print(info)

{'nocc': 40, 'nelec': 80, 'nao': 60, 'nmo': 60, 'nci': 10, 'min_occ': 37, 'max_occ': 40, 'min_vir': 41, 'max_vir': 47, 'nact': 60, 'actual_orbital_space': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]}


The data looks like this:

In [5]:
print(len(data))
print(data)

4
[[0.02228804527580758, 0.035988387049355045, 0.046914483113446766, 0.06919628091580611, 0.07872514791812135, 0.08426334938076513, 0.08584543015692184, 0.09154790342141046, 0.09546800926096066, 0.09640402778288193], [[[40, 41]], [[40, 42]], [[40, 43], [40, 44]], [[40, 44], [40, 46]], [[40, 45], [40, 46]], [[39, 41], [39, 42], [40, 46], [37, 41]], [[40, 46], [40, 45], [39, 41]], [[38, 41], [39, 42], [40, 47], [37, 41]], [[40, 47], [37, 41], [38, 42]], [[39, 42], [38, 41], [39, 43], [38, 42], [37, 41]]], [[-0.973537], [-0.971833], [0.938266, -0.247684], [-0.871286, 0.337412], [-0.868286, 0.355366], [0.798763, 0.313484, -0.28438, 0.278284], [-0.802833, -0.356074, -0.229352], [0.711404, -0.470105, 0.313769, 0.250202], [0.730334, -0.451852, 0.297089], [0.64084, 0.427694, -0.327131, -0.280067, -0.258855]], [['alp'], ['alp'], ['alp', 'alp'], ['alp', 'alp'], ['alp', 'alp'], ['alp', 'alp', 'alp', 'alp'], ['alp', 'alp', 'alp'], ['alp', 'alp', 'alp', 'alp'], ['alp', 'alp', 'alp'], ['alp', 'alp',

- `data[0]` - the **excited states energies** in atomic units (Hartree).
- `data[1]` - the **excited states configurations**. Its length is `number_of_states` and each element of that contain a list which is formed from one or multiple lists, each representing a single-particle excitation.
- `data[2]` - contains the **configuration interaction coefficients** related to each single-particle excitation in each excited states
- `data[3]` - contains the **spin** information

Let's examine them:

In [6]:
data[0]

[0.02228804527580758,
 0.035988387049355045,
 0.046914483113446766,
 0.06919628091580611,
 0.07872514791812135,
 0.08426334938076513,
 0.08584543015692184,
 0.09154790342141046,
 0.09546800926096066,
 0.09640402778288193]

In [7]:
data[1]

[[[40, 41]],
 [[40, 42]],
 [[40, 43], [40, 44]],
 [[40, 44], [40, 46]],
 [[40, 45], [40, 46]],
 [[39, 41], [39, 42], [40, 46], [37, 41]],
 [[40, 46], [40, 45], [39, 41]],
 [[38, 41], [39, 42], [40, 47], [37, 41]],
 [[40, 47], [37, 41], [38, 42]],
 [[39, 42], [38, 41], [39, 43], [38, 42], [37, 41]]]

Which means that the first excited state is composed of a dominant single particle excitation 

    40 (HOMO) -> 41 (LUOMO)
    
while the 3rd excited state is dominated by two configurations (each with amplitude more than `'tolerance': 0.02`):

    40 (HOMO) -> 43 (LUMO+2)
    40 (HOMO) -> 44 (LUMO+3)

In [8]:
data[2]

[[-0.973537],
 [-0.971833],
 [0.938266, -0.247684],
 [-0.871286, 0.337412],
 [-0.868286, 0.355366],
 [0.798763, 0.313484, -0.28438, 0.278284],
 [-0.802833, -0.356074, -0.229352],
 [0.711404, -0.470105, 0.313769, 0.250202],
 [0.730334, -0.451852, 0.297089],
 [0.64084, 0.427694, -0.327131, -0.280067, -0.258855]]

So, more specifically, this means that the third excitation is represented by:
$$
\lvert \Psi_3 \rangle = 0.938266 \lvert \Phi_{HOMO}^{LUMO} \rangle -0.247684 \lvert \Phi_{HOMO}^{LUMO+3} \rangle
$$

In [9]:
data[3]

[['alp'],
 ['alp'],
 ['alp', 'alp'],
 ['alp', 'alp'],
 ['alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp', 'alp']]

which means that all the excitations are in the $\alpha$ channel

## 2. Constructing the mapped SD basis and corresponding CSFs
<a name="2"></a>
[Back to TOC](#toc)

First, let's load the time-overlap of MOs

In [10]:
sample_matrix = sp.load_npz('St_ks_1200.npz')
#St_ks = sample_matrix[ks_active_space,:][:,ks_active_space]
print('sample_matrix.shape:', sample_matrix.shape)

sample_matrix.shape: (80, 80)


The dimensions of this matrix are determined by how it was computed in "step2"

In this case, the setups for "step2" were consistent with the current choice of the active space:

starting from `lowest_orbital = ks_homo_index - 19` and going all the way to `highest_orbital = ks_homo_index + 20`, where 
`ks_homo_index = 40`

The formula is: 
$$
N_{orbs} = \text{highest_orbital} - \text{lowest_orbital} + 1
$$

where 1 is because we also have HOMO itself

E.g. if ks_homo_index = 40, and lowest_orbital = 40 - 1 (HOMO-1) and highest_orbital = 40 + 1 (LUMO) that counts as 3 orbitals (HOMO-1, HOMO, LUMO)

For our setup this gives $N_{orbs} = (40 + 20) - (40 - 19) + 1 = 40$

But the dimensions of the St matrix are doubled because is formed of four block-matrices: diagonal matrices composed of alpha and beta MO time-overlaps and two zero off-diagonal block-matrices, which are formally the time-overlaps of the alpha and beta spin-orbitals (hence zero in the absence of SOC). 

We can check some properties of this matrix:

In [11]:
# Diagonal blocks are the same and close to the identity
print(sample_matrix[0:2,0:2], sample_matrix[40:42, 40:42])

# Off-diagonal blocks are zeroes
print(sample_matrix[0:2,40:42], sample_matrix[40:42, 0:2])

<Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 4 stored elements and shape (2, 2)>
  Coords	Values
  (0, 0)	(0.997212969940299+0j)
  (1, 0)	(-0.039998956054708844+0j)
  (0, 1)	(0.041006978090646864+0j)
  (1, 1)	(0.9986392465725624+0j) <Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 4 stored elements and shape (2, 2)>
  Coords	Values
  (0, 0)	(0.997212969940299+0j)
  (1, 0)	(-0.039998956054708844+0j)
  (0, 1)	(0.041006978090646864+0j)
  (1, 1)	(0.9986392465725624+0j)
<Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 0 stored elements and shape (2, 2)> <Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 0 stored elements and shape (2, 2)>


In [12]:
# list all the spatial orbitals (1-based) for which the MO time-overlaps are available
orbital_space = list(range(lowest_orbital, highest_orbital+1))
print(orbital_space)

[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]


In [13]:
# list the indices of the spatial orbital (1-based) across which the excitations are allowed
active_space = list(orbital_space) # select all the available ones

In [14]:
nelec = 40
ncore = nelec/2
GS = [ x for i in range(21, 41) for x in (i,-i)]
print(tuple(GS) )

(21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 40, -40)


Consider first excited state (HOMO -> LUMO)

In [15]:
indx = GS.index(40)
EX1 = list(GS); EX1[indx] = 41
EX2 = list(GS); EX2[indx] = 42
EX1 = tuple(EX1)
EX2 = tuple(EX2)
print(EX1, EX2)

(21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 41, -40) (21, -21, 22, -22, 23, -23, 24, -24, 25, -25, 26, -26, 27, -27, 28, -28, 29, -29, 30, -30, 31, -31, 32, -32, 33, -33, 34, -34, 35, -35, 36, -36, 37, -37, 38, -38, 39, -39, 42, -40)


In [16]:
help(interfaces.configs_and_T_matrix_singlet)

Help on function configs_and_T_matrix_singlet in module libra_py.citools.interfaces:

configs_and_T_matrix_singlet(configs0_raw: List[Tuple[int, ...]], active_space: List[int], orbital_space: List[int], nelec: int, S: int, Ms: int) -> Tuple[List[Tuple[int, ...]], scipy.sparse._coo.coo_matrix]
    Generate the minimal active-space configurations mapped to a given orbital space
    and the configuration-to-CSF transformation matrix for a CAS with given spin.
    
    Parameters
    ----------
    configs0_raw : list[tuple[int]]
        List of raw configurations from Libra/MOPAC (signed orbital indices).
    active_space : list[int]
        Orbitals defining the active space used to generate the minimal determinant basis.
    orbital_space : list[int]
        Orbital indices used for mapping configurations (output will be relative to this space).
    nelec : int
        Number of active electrons.
    S : int
        Total spin quantum number.
    Ms : int
        Spin projection quantum

<a name="configs_and_T_matrix_singlet-1"></a>

In [17]:
# In this example, we have all MO time-overlaps (starting from 1 and beyond the orbitals of active space)
# so the mapped basis looks trivial (using the same indices as of the raw configurations)
configs0_raw = [ tuple(GS), EX1, EX2 ]
S = 0
Ms = 0
max_unpaired = 0 # singlets only

mapped_basis, T = interfaces.configs_and_T_matrix_singlet(
    configs0_raw,  active_space,  orbital_space,
    nelec, S, Ms)
print(T)
print(mapped_basis)

<COOrdinate sparse matrix of dtype 'complex128'
	with 5 stored elements and shape (5, 3)>
  Coords	Values
  (0, 0)	(1+0j)
  (1, 1)	(0.7071067811865476+0j)
  (2, 1)	(-0.7071067811865476+0j)
  (3, 2)	(0.7071067811865476+0j)
  (4, 2)	(-0.7071067811865476+0j)
[(1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -18, 19, -19, 20, -20), (1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -18, 19, -19, -20, 21), (1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -18, 19, -19, 20, -21), (1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -18, 19, -19, -20, 22), (1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -

**Important addition:**

While the generated configurations are canonically-ordered, the calculations of the SD (and further) overlaps would miss the **excitation parity phase**. Thus, it needs to be computed separately and used later in the overlap calculations.

This is how we compute the phases:

In [18]:
nbas = len(mapped_basis)

Phases = [1.0] # ground state reference
for i in range(1, nbas):
    phase, new_det = sd.excitation_phase_from_mapping(mapped_basis[0], mapped_basis[i])
    Phases.append(phase)
    print(phase)

-1
1
-1
1


<a name="excitation_phase_from_mapping-1"></a>

In [19]:
help(sd.excitation_phase_from_mapping)

Help on function excitation_phase_from_mapping in module libra_py.citools.slatdet:

excitation_phase_from_mapping(det: 'List[int]', exc: 'List[int]') -> 'Tuple[int, List[int]]'
    Compute the fermionic phase for a single excitation between two
    Slater determinants given in canonical (sorted) form.
    
    Unlike position-based mapping, this version detects the excitation
    via *set differences*, making it robust to canonical reordering.
    
    Parameters
    ----------
    det : list of int
        Reference determinant (canonical order).
    
    exc : list of int
        Excited determinant (canonical order).
    
    Returns
    -------
    phase : int
        Fermionic phase factor (+1 or -1).
    
    new_det : list of int
        The excited determinant (canonical order).
    
    Raises
    ------
    ValueError
        If the excitation is not a single excitation.
    
    Notes
    -----
    Phase is computed as:
    
        phase = (-1)^(N_between)
    
    where N_

As we can see, the SD are now defined in terms of mapped orbital indices - with regard to what is actually present in the MO-basis time-overlap matrix we read above

In [20]:
T.todense()

matrix([[ 1.        +0.j,  0.        +0.j,  0.        +0.j],
        [ 0.        +0.j,  0.70710678+0.j,  0.        +0.j],
        [ 0.        +0.j, -0.70710678+0.j,  0.        +0.j],
        [ 0.        +0.j,  0.        +0.j,  0.70710678+0.j],
        [ 0.        +0.j,  0.        +0.j, -0.70710678+0.j]])

So, we have three CSFs in our basis (ground state and two excitations corresponding to 40->41 and 40->42 transitions) - columns of the T matrix,

and we have 5 "raw" SDs (rows of the T matrix) - the ground state determinant, single excitations 40->41 and 40->42 and their spin complements). 

Considering that HOMO is the orbital 40, and the virtual orbitals are 41 and 42. Including the ground-state determinant and single excitations with spin complements:

$$
\begin{aligned}
|D_0\rangle &= |40^\alpha, 40^\beta\rangle && \text{(ground state)} \\
|D_1\rangle &= |41^\alpha, 40^\beta\rangle && \text{(excitation 40 → 41)} \\
|D_2\rangle &= |41^\beta, 40^\alpha\rangle && \text{(spin complement)} \\
|D_3\rangle &= |42^\alpha, 40^\beta\rangle && \text{(excitation 40 → 42)} \\
|D_4\rangle &= |42^\beta, 40^\alpha\rangle && \text{(spin complement)}
\end{aligned}
$$

These are the **rows of the $T$ matrix**.

The singlet spin-adapted CSFs, including the ground state:

$$
\begin{aligned}
| \text{CSF}_0 \rangle &= |40^\alpha, 40^\beta\rangle && \text{(ground-state)} \\
| \text{CSF}_1 \rangle &= \frac{1}{\sqrt{2}} \Big( |41^\alpha, 40^\beta\rangle - |41^\beta, 40^\alpha\rangle \Big) && \text{(excitation 40 → 41)} \\
| \text{CSF}_2 \rangle &= \frac{1}{\sqrt{2}} \Big( |42^\alpha, 40^\beta\rangle - |42^\beta, 40^\alpha\rangle \Big) && \text{(excitation 40 → 42)}
\end{aligned}
$$

These are the **columns of the $T$ matrix**.

Mathematically, the CSFs are linear combinations of SDs:

$$
\begin{bmatrix}
| \text{CSF}_0 \rangle & | \text{CSF}_1 \rangle & | \text{CSF}_2 \rangle
\end{bmatrix}
=
\underbrace{
\begin{bmatrix}
|D_0\rangle \\
|D_1\rangle \\
|D_2\rangle \\
|D_3\rangle \\
|D_4\rangle
\end{bmatrix}}_{\text{rows of T}}
\underbrace{
\begin{bmatrix}
1 & 0 & 0 \\
0 & \frac{1}{\sqrt{2}} & 0 \\
0 & -\frac{1}{\sqrt{2}} & 0 \\
0 & 0 & \frac{1}{\sqrt{2}} \\
0 & 0 & -\frac{1}{\sqrt{2}}
\end{bmatrix}}_{T \text{ matrix}}
$$

## 3. Computing SD and CSF time-overlaps
<a name="3"></a>
[Back to TOC](#toc)

In [21]:
dets_A = list(mapped_basis)
dets_B = list(mapped_basis)

# sample_matrix is sparse, so we need to convert it to dense
st_mo = sample_matrix.todense()

print(st_mo.shape)
print(st_mo)

(80, 80)
[[ 0.99721297+0.j  0.04100698+0.j -0.00780894+0.j ...  0.        +0.j
   0.        +0.j  0.        +0.j]
 [-0.03999896+0.j  0.99863925+0.j -0.01163615+0.j ...  0.        +0.j
   0.        +0.j  0.        +0.j]
 [ 0.00748305+0.j  0.01131635+0.j  0.99906251+0.j ...  0.        +0.j
   0.        +0.j  0.        +0.j]
 ...
 [ 0.        +0.j  0.        +0.j  0.        +0.j ...  0.99077979+0.j
  -0.03785856+0.j  0.03374368+0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j ... -0.03735992+0.j
  -0.99806703+0.j -0.03003087+0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j ... -0.0320201 +0.j
  -0.0347664 +0.j  0.98065293+0.j]]


<a name="slater_overlap_matrix-1"></a>

In [22]:
st_sd = sd.slater_overlap_matrix(dets_A, dets_B, st_mo, complex_valued=False, phases_A=Phases, phases_B=Phases)

/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/citools/slatdet.py:351: ComplexWarning: Casting complex values to real discards the imaginary part
  S_AB[i, j] = np.linalg.det(S_a) * np.linalg.det(S_b) * ph_A[i] * ph_B[j]


In [23]:
print(st_sd)

[[ 9.93595120e-01 -1.52251643e-02  1.52251643e-02 -9.55270033e-03
   9.55270033e-03]
 [ 1.54574818e-02  9.93460231e-01  2.36859758e-04  1.50389576e-02
   1.48612537e-04]
 [-1.54574818e-02  2.36859758e-04  9.93460231e-01  1.48612537e-04
   1.50389576e-02]
 [ 9.45620098e-03 -1.53016682e-02  1.44900282e-04  9.93439942e-01
   9.09145510e-05]
 [-9.45620098e-03  1.44900282e-04 -1.53016682e-02  9.09145510e-05
   9.93439942e-01]]


The difference that is proportional to the NAC betweend SDs shows that 
$$
\langle D_0 | \frac{\partial}{\partial t} | D_1 \rangle = - \langle D_0 | \frac{\partial}{\partial t} | D_2 \rangle
$$
and 
$$
\langle D_0 | \frac{\partial}{\partial t} | D_3 \rangle = - \langle D_0 | \frac{\partial}{\partial t} | D_4 \rangle
$$
as it should be (can be shown analytically)

In [24]:
print(st_sd.T - st_sd)

[[ 0.00000000e+00  3.06826460e-02 -3.06826460e-02  1.90089013e-02
  -1.90089013e-02]
 [-3.06826460e-02  0.00000000e+00  0.00000000e+00 -3.03406257e-02
  -3.71225464e-06]
 [ 3.06826460e-02  0.00000000e+00  0.00000000e+00 -3.71225464e-06
  -3.03406257e-02]
 [-1.90089013e-02  3.03406257e-02  3.71225464e-06  0.00000000e+00
   0.00000000e+00]
 [ 1.90089013e-02  3.71225464e-06  3.03406257e-02  0.00000000e+00
   0.00000000e+00]]


Furthermore, one can show that the corresponding time-derivatives of SDs are related to time-derivatives of MOs, as was shown in:

> Akimov A. V.; Prezhdo O. V. "The PYXAID program for non-adiabatic molecular dynamics in condensed matter systems"  J. Comp. Theory Comput. 2013, 9, 4959 

$$
\langle D_0 | \frac{\partial}{\partial t} | D_1 \rangle \sim \langle HOMO | \frac{\partial}{\partial t} | LUMO \rangle
$$

and 

$$
\langle D_0 | \frac{\partial}{\partial t} | D_3 \rangle \sim \langle HOMO | \frac{\partial}{\partial t} | LUMO+1 \rangle
$$

As you can see, the numerical values are equal, but the overall sign may be different (yet relative signs are preserved)

In [25]:
# 19 - HOMO - index of HOMO (40) in the sub-set of orbitals included in St_ks
# 
act_st_mo = st_mo[ 19:22, 19:22 ]
print("MO time-overlaps of HOMO, LUMO, LUMO+1")
print(act_st_mo)
print("NACs of HOMO, LUMO, LUMO+1")
print( (act_st_mo.T - act_st_mo) )

MO time-overlaps of HOMO, LUMO, LUMO+1
[[ 0.99973896+0.j  0.01533404+0.j  0.0096025 +0.j]
 [-0.01553914+0.j  0.99965178+0.j  0.01513334+0.j]
 [-0.00952422+0.j -0.01539659+0.j  0.9996253 +0.j]]
NACs of HOMO, LUMO, LUMO+1
[[ 0.        +0.j -0.03087318+0.j -0.01912672+0.j]
 [ 0.03087318+0.j  0.        +0.j -0.03052993+0.j]
 [ 0.01912672+0.j  0.03052993+0.j  0.        +0.j]]


Now, let's compute the time-overlap in the SCF basis.

Note, the matrix T is sparse, and we don't need to convert it to the dense format for the transformation to work

In [26]:
st_csf = T.T @ st_sd @ T

In [27]:
print(st_csf)

[[ 0.99359512+0.j -0.02153163+0.j -0.01350956+0.j]
 [ 0.02186018+0.j  0.99322337+0.j  0.01489035+0.j]
 [ 0.01337309+0.j -0.01544657+0.j  0.99334903+0.j]]


Indeed, keeping all elements (dense format) yields numerically the same results:

In [28]:
T_dense = T.todense()
st_csf_dense = T_dense.T @ st_sd @ T_dense
print(st_csf_dense)

[[ 0.99359512+0.j -0.02153163+0.j -0.01350956+0.j]
 [ 0.02186018+0.j  0.99322337+0.j  0.01489035+0.j]
 [ 0.01337309+0.j -0.01544657+0.j  0.99334903+0.j]]


Now, let's compare it to the what we have been doing for long time - using the time-overlaps directly in the basis of "raw" SDs.

To do this, we just redefine the `mapped_basis` to exclude the spin-complement configurations

In [29]:
mapped_basis_red = [mapped_basis[i] for i in (0, 1, 3)]
Phases1 = [Phases[i] for i in [0,1,3] ]
dets_A_red = list(mapped_basis_red)
dets_B_red = list(mapped_basis_red)

st_sd_raw = sd.slater_overlap_matrix(dets_A_red, dets_B_red, st_mo, complex_valued=False, phases_A=Phases1, phases_B=Phases1)

In [30]:
print(st_sd_raw)

[[ 0.99359512 -0.01522516 -0.0095527 ]
 [ 0.01545748  0.99346023  0.01503896]
 [ 0.0094562  -0.01530167  0.99343994]]


which gives pretty close results, but not exactly the same

Let's now compare what would be proportional to scalar NACs

In [31]:
print("Spin-adapted:")
print(st_csf.T - st_csf)

print("Raw:")
print(st_sd_raw.T - st_sd_raw)

Spin-adapted:
[[ 0.        +0.j  0.04339181+0.j  0.02688265+0.j]
 [-0.04339181+0.j  0.        +0.j -0.03033691+0.j]
 [-0.02688265+0.j  0.03033691+0.j  0.        +0.j]]
Raw:
[[ 0.          0.03068265  0.0190089 ]
 [-0.03068265  0.         -0.03034063]
 [-0.0190089   0.03034063  0.        ]]


In agreement with the previous results:

> Shakiba, M.; Stippel, E.; Li, W.; Akimov, A. V.* "Nonadiabatic Molecular Dynamics with Extended Density Functional Tight-Binding: Application to Nanocrystals and Periodic Solids" J. Chem. Theory Comput. 2022  18, 5157-5180

the NACs between ground state and single excitations are the factor of $\sqrt 2$ larger than the corresponding NACs in the MO basis:

$$
\langle CSF_0 | \frac{\partial}{\partial t} | CSF_1 \rangle = \sqrt 2 \langle D_0 | \frac{\partial}{\partial t} | D_1 \rangle
$$

$$
\langle CSF_0 | \frac{\partial}{\partial t} | CSF_2 \rangle = \sqrt 2 \langle D_0 | \frac{\partial}{\partial t} | D_3 \rangle
$$

but the couplings between single excitations are comparable to those without spin-adaptation:

$$
\langle CSF_1 | \frac{\partial}{\partial t} | CSF_2 \rangle = \langle D_1 | \frac{\partial}{\partial t} | D_3 \rangle
$$

In [32]:
0.03068265 * np.sqrt(2) # should be close to 0.04339181

np.float64(0.04339181975954685)

In [33]:
0.0190089 * np.sqrt(2) # should be close to 0.02688265

np.float64(0.026882644185793926)

## 4. Computing CI time-overlaps
<a name="4"></a>
[Back to TOC](#toc)

### 4.1. Organizing basis and amplitudes
<a name="4.1"></a>
[Back to TOC](#toc)

Now, when we know how to deal with small bases of SD and SCF, let's consider a realistic example.

Assume, we are interested in computing the time-overlaps for all the above 10 CI states + the ground state. 

We first, need to define the basis of unique excitations

In [34]:
data[1]

[[[40, 41]],
 [[40, 42]],
 [[40, 43], [40, 44]],
 [[40, 44], [40, 46]],
 [[40, 45], [40, 46]],
 [[39, 41], [39, 42], [40, 46], [37, 41]],
 [[40, 46], [40, 45], [39, 41]],
 [[38, 41], [39, 42], [40, 47], [37, 41]],
 [[40, 47], [37, 41], [38, 42]],
 [[39, 42], [38, 41], [39, 43], [38, 42], [37, 41]]]

In [35]:
unique_sds = []
for ex in data[1]:
    for sd in ex:
        if sd not in unique_sds:
            unique_sds.append(sd)
print(unique_sds)

[[40, 41], [40, 42], [40, 43], [40, 44], [40, 46], [40, 45], [39, 41], [39, 42], [37, 41], [38, 41], [40, 47], [38, 42], [39, 43]]


## Important!  the unique basis should be determined for the both snapshots, if time-overlap will be computed

This is because the CI states may be expressed in terms of different sets of Slater determiants at different geometries. 

This is because of the threshold for CI coefficients. If it is set to zero, all the terms would be avaialable, so the basis of the SD configurations would be constant at all times.

In [36]:
data[2]

[[-0.973537],
 [-0.971833],
 [0.938266, -0.247684],
 [-0.871286, 0.337412],
 [-0.868286, 0.355366],
 [0.798763, 0.313484, -0.28438, 0.278284],
 [-0.802833, -0.356074, -0.229352],
 [0.711404, -0.470105, 0.313769, 0.250202],
 [0.730334, -0.451852, 0.297089],
 [0.64084, 0.427694, -0.327131, -0.280067, -0.258855]]

Let's map the CI coefficients to this basis:

In [37]:
unique_sds.index([40, 41])

0

In [38]:
# In general, 
#nstates = len(data[1]) + 1 # including the ground state

# But here, we will consider only 3 states - the ground and two excited states - they are, luckily expressed in terms of only 3 
# determinants considered above
nstates = 3

# In general
#nsd = len(unique_sds) + 1 # including the reference determinant
nsd = 3

C = np.zeros((nsd, nstates), dtype=np.float128)
C[0,0] = 1.0 # ground state
for i in range(nstates-1):
    print(F"excited state {i+1}:")
    nconf = len(data[1][i]) # how many configurations there are in state i+1 (excited state i)
    for j in range(nconf):
        det = data[1][i][j]
        j_glob = unique_sds.index(det)  # find the index of this conf `j` in the unique basis
        print(F"  SD: |{j_glob}> {data[2][i][j]} {det}")
        
        C[j_glob+1, i+1] = data[2][i][j] # inrement j_glob by 1 to account for the reference det
                                         # increment i by 1 to account for the ground state

print(C)

excited state 1:
  SD: |0> -0.973537 [40, 41]
excited state 2:
  SD: |1> -0.971833 [40, 42]
[[ 1.        0.        0.      ]
 [ 0.       -0.973537  0.      ]
 [ 0.        0.       -0.971833]]


### 4.2. Transformation to the CI basis
<a name="4.2"></a>
[Back to TOC](#toc)

So we are finally ready to do it

In [39]:
st_ci = C.T @ st_csf @ C
print(st_ci)

[[ 0.99359512+0.j  0.02096184+0.j  0.01312903+0.j]
 [-0.02128169+0.j  0.94135158+0.j  0.01408798+0.j]
 [-0.01299641+0.j -0.01461424+0.j  0.93817781+0.j]]


In [40]:
st_ci_raw = C.T @ st_sd_raw @ C
print(st_ci_raw)

[[ 0.99359512  0.01482226  0.00928363]
 [-0.01504843  0.94157607  0.01422859]
 [-0.00918985 -0.01447714  0.93826367]]


Compare NACs:

In [41]:
print("Spin-adapted:")
print(st_ci.T - st_ci)

print("Raw:")
print(st_ci_raw.T - st_ci_raw)

Spin-adapted:
[[ 0.        +0.j -0.04224354+0.j -0.02612544+0.j]
 [ 0.04224354+0.j  0.        +0.j -0.02870222+0.j]
 [ 0.02612544+0.j  0.02870222+0.j  0.        +0.j]]
Raw:
[[ 0.         -0.02987069 -0.01847348]
 [ 0.02987069  0.         -0.02870573]
 [ 0.01847348  0.02870573  0.        ]]
